# 📨 MessagesState — Pre-built State for Chat Graphs

## Learning Objectives
In this notebook, you will learn:
1. **`MessagesState`** — the pre-built base class that gives you `messages` with `add_messages` for free
2. **Extending MessagesState** — adding custom state variables alongside messages
3. **Multi-node workflows** — chaining user, AI, and counter nodes

## Prerequisites
- `langgraph`, `langchain-core` installed
- Understanding of `add_messages` reducer (notebook `05`)

---
## 🔧 Part 1: Environment Setup

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
from typing import Annotated, List, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages

print("✅ Imports loaded successfully!")

✅ Imports loaded successfully!


---
## 📋 Part 2: Define the State

By subclassing `MessagesState`, we inherit a `messages` key with the
`add_messages` reducer already configured. We only need to add our custom fields.

> **Key Insight**: `MessagesState` is equivalent to writing:
> ```python
> class MyState(TypedDict):
>     messages: Annotated[list[BaseMessage], add_messages]
> ```

In [2]:
# ============================================================================
# STATE DEFINITION: Extend MessagesState with custom fields
# ============================================================================
class MyGraphState(MessagesState):
    turn_count: int

print("✅ State schema defined!")

✅ State schema defined!


---
## ⚙️ Part 3: Define the Nodes

We create three nodes that form a simple turn-based conversation:
1. **user_node** — simulates a human message
2. **ai_node** — simulates an AI response based on the last message
3. **counter_node** — increments the turn counter

In [3]:
# ============================================================================
# NODE DEFINITIONS: user, AI, and counter nodes
# ============================================================================
def user_node(state: MyGraphState) -> dict:
    """Simulates a human message."""
    print("Executing user_node...")
    return {"messages": HumanMessage(content="What's the weather like today?")}


def ai_node(state: MyGraphState) -> dict:
    """Simulates an AI response using the last human message."""
    print("Executing ai_node...")
    last_human_message = state["messages"][-1]

    response_content = (
        f"I've received your query: '{last_human_message.content}'. "
        "I can't tell you the weather right now, but I can confirm that my code is working!"
    )
    return {"messages": AIMessage(content=response_content)}


def counter_node(state: MyGraphState) -> dict:
    """Increments the turn counter."""
    print("Executing counter_node...")
    return {"turn_count": state["turn_count"] + 1}

print("✅ Nodes defined!")

✅ Nodes defined!


---
## 🔗 Part 4: Build & Run the Graph

The flow is sequential: `START → user_input → ai_response → increment_counter → END`.

In [4]:
# ============================================================================
# GRAPH CONSTRUCTION: Build, wire, compile, and invoke
# ============================================================================
graph = StateGraph(MyGraphState)

graph.add_node("user_input", user_node)
graph.add_node("ai_response", ai_node)
graph.add_node("increment_counter", counter_node)

graph.add_edge(START, "user_input")
graph.add_edge("user_input", "ai_response")
graph.add_edge("ai_response", "increment_counter")
graph.add_edge("increment_counter", END)

agent = graph.compile()

print("✅ Graph compiled!")

✅ Graph compiled!


In [5]:
# ============================================================================
# EXECUTION: Invoke the graph
# ============================================================================
initial_state = {"turn_count": 0}

final_state = agent.invoke(initial_state)

print("\n--- Final State of the Graph ---")
print(final_state)

Executing user_node...
Executing ai_node...
Executing counter_node...

--- Final State of the Graph ---
{'messages': [HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}, id='3f2d4fdf-f806-4123-b380-3329be86e005'), AIMessage(content="I've received your query: 'What's the weather like today?'. I can't tell you the weather right now, but I can confirm that my code is working!", additional_kwargs={}, response_metadata={}, id='2fac1888-5e5c-44fb-ac9c-0004edd89072', tool_calls=[], invalid_tool_calls=[])], 'turn_count': 1}


{'messages': [HumanMessage(content="What's the weather like today?", additional_kwargs={}, response_metadata={}, id='3f2d4fdf-f806-4123-b380-3329be86e005'), AIMessage(content="I've received your query: 'What's the weather like today?'. I can't tell you the weather right now, but I can confirm that my code is working!", additional_kwargs={}, response_metadata={}, id='2fac1888-5e5c-44fb-ac9c-0004edd89072', tool_calls=[], invalid_tool_calls=[])], 'turn_count': 1}


In [6]:
for elem in final_state["messages"]:
    print(f"{elem.__class__.__name__}: {elem.content}")

HumanMessage: What's the weather like today?
AIMessage: I've received your query: 'What's the weather like today?'. I can't tell you the weather right now, but I can confirm that my code is working!


---
## 📝 Summary

In this notebook, we learned:

### 1. MessagesState
- Pre-built class that provides `messages: Annotated[list, add_messages]`
- Subclass it to add custom fields like `turn_count`

### 2. Multi-Node Pipelines
- Chain multiple nodes in a sequential workflow
- Each node updates different parts of the shared state

### 3. Mixing Message and Non-Message State
- `messages` uses the `add_messages` reducer (appends)
- `turn_count` uses the default reducer (replaces)

### Next Steps
- Explore **Node arguments** and runtime config (notebook `07-nodes`)
- Learn **Edges** and conditional routing (notebook `08-edges`)